In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 4.8 A Taste of Curved Spacetime

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume IV — Special Relativity",
    number="4.8",
    title="A Taste of Curved Spacetime",
    blurb="The capstone: gravity as geometry. The equivalence principle makes "
    "acceleration and gravity locally indistinguishable, clocks run slow "
    "in a gravitational well, and a test particle near a black hole traces "
    "an orbit no Newtonian calculation can produce — all computed, even "
    "where the full theory lies beyond a first course.",
    difficulty="advanced",
    estimate="200–260 min",
)

## About this notebook

This is the longest and most ambitious notebook of the course so far, and the capstone of
Volume IV. It is a *taste* of general relativity, not a course in it. We will state a few
equations that the full theory derives and we cannot yet — the Schwarzschild metric, the
orbit equation, the null geodesic of a light ray — and then we will **compute with them**.
That is not a compromise but the point: engaging with physics beyond one's current
theoretical reach, through computation, is a genuine and valuable way of doing physics.
Where a result lies past what a first pass can derive, we will say so plainly and then hand
over the computational handle, because the machinery built since Volume 0 — integrating an
ODE, plotting an orbit, measuring an angle — is exactly what lets us reach it.

The notebook has two movements. **Movement I** is the conceptual on-ramp: the equivalence
principle, which makes gravity a feature of spacetime geometry rather than a force, and its
immediate consequence, that clocks run slow in a gravitational well — an effect your phone's
GPS must correct for every day. **Movement II** is curved spacetime computed: the
Schwarzschild geometry around a black hole, where we integrate a test particle's orbit and
watch it **precess** into a rosette no Newtonian gravity can produce, recover Mercury's
famous $43''$ per century, locate the innermost stable orbit and the photon sphere, drop a
particle through the horizon, and bend a ray of starlight.

A word on units. Movement II works in **geometric units** where lengths are measured in the
Schwarzschild radius (equivalently $G=c=1$), a choice we motivate at length, partly as a
correctness practice that traces straight back to the floating-point lesson of [§0.1](../00-foundations/floating-point.ipynb).
Movement I stays in SI. One central image, the precessing orbit, is genuine motion and is
animated, as is the plunge-versus-escape comparison; the rest are clean stills.

> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the Pound–Rebka redshift $gh/c^2$; the Schwarzschild clock rate
> $\sqrt{1-2M/r}$; the GPS offset; the Sun's $2.954\,$km Schwarzschild radius; a precessing
> (non-closing) orbit; Mercury's $43''$/century; the ISCO at $6M$ and photon sphere at $3M$;
> the horizon-crossing plunge; the $4M/b$ light deflection; the black-hole shadow at
> $3\sqrt3\,M$. A ✓ is strong evidence; a ✗ is a prompt to *locate the discrepancy*, not a
> verdict.
>
> **Scope.** A computational *taste* of general relativity: we state the metric and geodesic
> equations with physical motivation and compute their predictions, rather than deriving them
> from the field equations (that is GR proper). See Hartle, *Gravity*; Misner, Thorne &
> Wheeler, *Gravitation*; Schutz, *A First Course in General Relativity*; and [§4.7](relativistic-lagrangian-fields.ipynb)
> (extremize proper time), [§0.1](../00-foundations/floating-point.ipynb) (floating point and nondimensionalization), and [§2.9](../02-classical-mechanics/lagrange-points.ipynb) (the
> nondimensional CR3BP).

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.optimize import brentq
from scipy.signal import argrelmax

from ecp import draw, validate
from ecp.animate import show

from scipy.constants import c as C_LIGHT  # speed of light, m/s
from scipy.constants import G as G_NEWTON  # gravitational constant, m^3/(kg·s^2)

M_SUN = 1.98892e30  # solar mass, kg
ARCSEC = 180.0 / np.pi * 3600.0  # radians → arcseconds
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT

# Movement I — The equivalence principle (gravity is geometry)

## Theory in brief

### The equivalence principle

Einstein's "happiest thought" was that a person in free fall feels no gravity. Promoted to a
principle, this says a uniformly accelerated frame is **locally indistinguishable** from a
uniform gravitational field,

```{math}
:label: eq-equivalence
\text{(accelerated frame, acceleration } g) \ \simeq\ \text{(uniform gravitational field } g),
```

which forces inertial mass to equal gravitational mass and, more deeply, means gravity can
always be transformed away locally by falling freely. Gravity is therefore not a force but a
property of spacetime geometry; free fall is "straight" (geodesic) motion through it.

### Gravitational time dilation

A clock deeper in a gravitational potential runs slower. In the weak field its rate is
$1+\Phi/c^2$, and in the exact Schwarzschild field of a mass $M$,

```{math}
:label: eq-grav-time-dilation
\frac{d\tau}{dt}=\sqrt{1-\frac{2GM}{rc^2}}=\sqrt{1-\frac{2M}{r}},
```

(the second form in units $G=c=1$), which falls to zero as $r\to2M$, the horizon
(Hartle, *Gravity*, derives both the weak-field and exact rates in full).

### Gravitational redshift

Light climbing out of a potential well loses energy and reddens. In the weak field the
fractional shift is

```{math}
:label: eq-redshift
\frac{\Delta f}{f}\approx\frac{gh}{c^2},
```

the effect Pound and Rebka measured in 1959 by sending gamma rays up a $22.5\,$m tower.

## Exercise 1 — The equivalence principle, quantified

The equivalence principle already predicts a number. Picture a photon rising a height $h$ in
a gravitational field $g$. By the principle, working in the field is the same as working in a
frame accelerating upward at $g$; in the time $h/c$ the photon takes to rise, the receiver at
the top has accelerated to a speed $gh/c$ away from the source, so it measures a Doppler
redshift $\Delta f/f=-gh/c^2$ {eq}`eq-redshift`. Light loses frequency climbing out of a
well. Pound and Rebka confirmed this tiny shift in 1959.

**This exercise (worked).** For the Pound–Rebka tower ($g=9.81\,$m/s², $h=22.5\,$m), compute
the fractional redshift $gh/c^2$ as plain `numpy` arithmetic and confirm it is about
$2.46\times10^{-15}$. Then re-derive it by the equivalence-principle route itself: when the
light arrives the receiver has acquired speed $\beta=gh/c^2$, so the *relativistic Doppler
factor* $\sqrt{(1-\beta)/(1+\beta)}$ must reproduce the same shift to first order —
evaluate it with `numpy.log1p` and `numpy.expm1`, because a shift of $10^{-15}$ computed
naively as $1-\sqrt{\cdots}$ would drown in float cancellation (the lesson of [§0.1](../00-foundations/floating-point.ipynb)),
and confirm the two routes agree.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    doppler_shift,
    df_f,
    "the equivalence-principle Doppler factor reproduces the gh/c² redshift to first order",
    rtol=1e-9,
)

## Exercise 2 — Gravitational time dilation in the Schwarzschild field

The redshift is one face of a deeper fact: clocks themselves run slow in a gravitational
well. Around a spherical mass the exact rate of a clock at radius $r$, relative to a distant
one, is $\sqrt{1-2M/r}$ {eq}`eq-grav-time-dilation` (working in units $G=c=1$, so $r$ is
measured in units of the mass $M$). The effect is gentle far out but becomes dramatic near
the horizon at $r=2M$, where the rate falls to zero: to a distant observer, a clock lowered
toward the horizon appears to freeze ({numref}`fig-gr-clockrate`).

**This exercise (worked).** Tabulate the clock rate $\sqrt{1-2M/r}$ at $r=6M$, $3M$, and
$2.1M$ as explicit `numpy` arithmetic and confirm the exact values $\sqrt{2/3}$ and
$1/\sqrt3$ at $6M$ and $3M$. Then check the exact formula against the *independent*
weak-field physics of Exercise 1 — far from the mass the rate must approach $1+\Phi/c^2=
1-M/r$ — by evaluating both at $r=10^4M$ and confirming they agree to $O(1/r^2)$. Plot the
rate against radius down to the horizon.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.array([clock_rate[0], clock_rate[1], rate_exact_far]),
    np.array([np.sqrt(2.0 / 3.0), 1.0 / np.sqrt(3.0), rate_weak_far]),
    "the clock rate hits its exact values at 6M and 3M and the weak-field 1−M/r far out",
    rtol=1e-7,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — GPS: relativity in daily life

Gravitational time dilation is not exotic; it is engineered around in the device in your
pocket. A GPS satellite orbits at an altitude of about $20\,200\,$km, higher in Earth's
potential well than the ground, so its clock runs *fast* relative to the surface by the
weak-field rate difference $\Delta\Phi/c^2$ {eq}`eq-grav-time-dilation`. This gravitational
effect amounts to roughly $+46\,\mu$s per day, partly offset by a special-relativistic
slowing from the satellite's orbital speed ([§4.1](crisis-and-postulates.ipynb)), for a net of about $+38\,\mu$s/day. Left
uncorrected, that error would corrupt positions by some $10\,$km per day.

**This exercise (worked).** Compute the gravitational rate difference between a GPS satellite
and the surface from the Newtonian potential $\Phi=-GM/r$, convert to microseconds per day,
and confirm it is about $+46\,\mu$s/day; then subtract the special-relativistic velocity term
$-\tfrac12v^2/c^2$ to get the net offset.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    grav_us_day,
    46.0,
    "GPS satellite clocks run fast by ~46 µs/day from gravitational time dilation",
    rtol=5e-2,
)

## Exercise 4 — Gravity as geometry

The equivalence principle says gravity can be transformed away locally by free fall — so how
is *real* gravity different from mere acceleration? The answer is **tidal** effects, and they
are the true, non-removable signature of spacetime curvature. In a genuinely uniform field,
two nearby free-fallers accelerate identically and their separation never changes; the field
is entirely transformed away in the falling frame. But a real gravitational field is not
uniform — it points toward the mass and weakens with distance — so two free-fallers at
slightly different radii accelerate *differently* and drift apart. That differential
acceleration, the **tidal field**, cannot be removed by any choice of frame, and it is
exactly what general relativity calls curvature. Free fall is geodesic motion; tidal drift is
the geodesics curving relative to one another.

**This exercise (student).** Compare two free-fallers separated radially by $\delta r=1\,$m
near Earth's surface. Confirm with a `numpy` comparison that in a *uniform* field their
relative acceleration is exactly zero, while in the real $1/r^2$ field the differential
(tidal) acceleration $GM(1/r_1^2-1/r_2^2)\approx2GM\,\delta r/r^3$ is nonzero — the
unremovable signature of curvature.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    abs(uniform_relative) < 1e-12
    and tidal_relative > 0
    and np.isclose(tidal_relative, tidal_estimate, rtol=1e-3),
    "a uniform field is removable (zero tidal effect), but a real field has nonzero tidal curvature",
)

# Movement II — Curved spacetime, computed (black holes)

## Theory in brief

### The Schwarzschild metric

The geometry of spacetime around a spherical mass $M$ is the **Schwarzschild metric**,

```{math}
:label: eq-schwarzschild
ds^2=-\Big(1-\frac{2M}{r}\Big)c^2dt^2+\Big(1-\frac{2M}{r}\Big)^{-1}dr^2+r^2d\phi^2,
```

(in the orbital plane, units $G=c=1$). *You may not yet have the machinery to derive this
from Einstein's field equations — that is general relativity proper, a course in itself.
But you do have the machinery to compute with it.* (Schutz, *A First Course in General
Relativity*, Ch. 10, carries the derivation out in full.) It defines the Schwarzschild radius
$r_s=2GM/c^2$, the horizon at $r=2M$.

### The orbit equation

A massive test particle's orbit obeys, with $u=1/r$,

```{math}
:label: eq-orbit
\frac{d^2u}{d\phi^2}+u=\frac{M}{L^2}+3Mu^2,
```

identical to the Newtonian orbit equation *except* for the $3Mu^2$ term — the general-
relativistic correction (Hartle, *Gravity*, Ch. 9, derives it from the geodesics of the
metric). Without it, orbits are closed Kepler ellipses; with it, they
precess. The motion is governed by an **effective potential**, exactly the tool of Volumes I
and II,

```{math}
:label: eq-special-radii
V_{\rm eff}(r)=\Big(1-\frac{2M}{r}\Big)\Big(1+\frac{L^2}{r^2}\Big),
```

whose critical structure sets the special radii: the **innermost stable circular orbit**
(ISCO) at $r=6M$ and the **photon sphere** at $r=3M$.

### Light bending

A light ray follows a null geodesic, $d^2u/d\phi^2+u=3Mu^2$, and a ray grazing the mass with
impact parameter $b$ is deflected by

```{math}
:label: eq-light-bending
\Delta\phi\approx\frac{4M}{b},
```

twice the Newtonian "falling corpuscle" value — the prediction Eddington's 1919 eclipse
expedition confirmed (Hartle, *Gravity*, Ch. 9, carries the weak-field computation out in
full).

### Nondimensionalization — why we work in units of $M$

Movement II measures every length in the Schwarzschild radius, equivalently setting $G=c=1$,
and this deserves a careful justification, because it is a transferable technique worth
reaching for whenever a problem spans many magnitudes.

1. **Only ratios matter.** An orbit at $r/M=20$ behaves identically whether the central mass
   is a star or a supermassive galaxy core; the physics depends on the dimensionless ratio
   $r/M$, not on $r$ and $M$ separately. Nondimensionalizing exposes the true, smaller set of
   parameters (a Buckingham-$\pi$ reduction): the orbit shape depends only on $L/M$, not on
   $L$, $M$, $G$, and $c$ independently.
2. **Numerical conditioning — the [§0.1](../00-foundations/floating-point.ipynb) callback.** In raw SI, a Schwarzschild
   quantity mixes $G\approx6.7\times10^{-11}$, $M\approx2\times10^{30}$, and $c^2\approx9
   \times10^{16}$ — magnitudes spanning some $40$ orders in one expression. Floating point
   (the $\varepsilon$ of [§0.1](../00-foundations/floating-point.ipynb)) carries only $\sim16$ significant digits, so combining
   such disparate magnitudes *loses precision* and risks overflow. Working in units of $M$
   keeps every number $O(1)$, where floating point is most accurate. This is a **correctness**
   practice, not mere tidiness.
3. **Round-trip discipline.** Compute the physical scale $r_s$ once in SI, work entirely
   dimensionless, then convert the answer back. We do exactly this for Mercury below, turning
   a dimensionless precession into the famous $43''$ per century.

We have quietly nondimensionalized before — the restricted three-body problem of [§2.9](../02-classical-mechanics/lagrange-points.ipynb), the
relaxation grids of [§3.4](../03-electrodynamics/laplace-poisson.ipynb) — but here we name it and justify it as a deliberate technique.

## Exercise 5 — Nondimensionalization, end to end

Before computing orbits, we fix the bridge between physical and working units, and we do it
on a real object. The Schwarzschild radius $r_s=2GM/c^2$ {eq}`eq-schwarzschild` is the one
length scale the geometry provides; in units of it (or equivalently $G=c=1$, lengths in units
of $M$) every Schwarzschild quantity becomes a pure number. The horizon sits at $r=2M$, the
photon sphere at $3M$, the ISCO at $6M$ — statements with no $G$, $c$, or kilograms in them.

**This exercise (worked).** Compute the Sun's Schwarzschild radius $2GM_\odot/c^2$ in SI
metres as plain `numpy` arithmetic, confirm it is about $2.954\,$km, and note the three
motivations above (ratios, conditioning, round-trip) that make the units-of-$M$ scheme both
natural and numerically sound.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    r_s_sun,
    2954.0,
    "the Sun's Schwarzschild radius is 2.954 km (the published value, in metres)",
    rtol=1e-3,
)

## Exercise 6 — The precessing orbit

Now the centerpiece. A test particle near a black hole obeys the orbit equation $d^2u/d\phi^2
+u=M/L^2+3Mu^2$ {eq}`eq-orbit`, and we are going to compute its path. *Here is the honest
situation: you may not yet be able to derive this equation from the Einstein field equations
— that is general relativity proper. But you have everything needed to compute with it.* Take
it as given, integrate it with `solve_ivp`, plot $r=1/u$ in the orbital plane, and measure
how far the perihelion advances each orbit. The extra $3Mu^2$ term, absent from Newton, makes
the orbit fail to close: instead of a fixed ellipse it traces a slowly turning **rosette**
({numref}`fig-gr-rosette`). What follows is a real prediction of general relativity, obtained
by integrating an ODE.

**This exercise (worked).** In units $M=1$, choose the angular momentum so the orbit runs
between perihelion $15M$ and aphelion $21M$ (the apsides exaggerated for visibility), integrate
the orbit equation with `scipy.integrate.solve_ivp` (DOP853, small `max_step`), and confirm the
orbit precesses — the perihelion advances by a definite angle each revolution rather than
closing. The animation traces the rosette around the hole.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    advance > 0.1 and 14.0 < r_orbit.min() < 16.0 and 20.0 < r_orbit.max() < 22.0,
    "the GR term 3Mu² makes the orbit precess — a rosette, not a closed ellipse",
)
validate.check(
    1.0 < advance / advance_weak < 1.8,
    "the measured advance agrees with the weak-field 6πM/p up to the strong-field enhancement",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Mercury, the real number

The exaggerated orbit of Exercise 6 was for the eye; the real solar-system effect is tiny but
was the first triumph of general relativity. Mercury's perihelion was known to advance by an
amount that Newtonian gravity, even after accounting for every other planet, could not fully
explain — a stubborn anomaly of about $43''$ per century. General relativity's weak-field
precession per orbit is $\Delta\phi=6\pi GM/(a(1-e^2)c^2)$ {eq}`eq-orbit`, and evaluating it
for Mercury, then converting to arcseconds per century, lands exactly on the missing number.
This is the round-trip discipline in action: a dimensionless formula, evaluated for a real
planet, converted back to physical units.

**This exercise (worked).** Using Mercury's orbit ($a=5.79\times10^{10}\,$m, $e=0.2056$,
period $88.0\,$days), compute the GR precession per orbit, multiply by the number of orbits
per century, convert radians to arcseconds, and confirm it is about $43''$/century.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    precession_arcsec_century,
    43.0,
    "GR predicts Mercury's 43″/century perihelion precession — the observed anomaly",
    rtol=5e-2,
)

**Closing a thread that runs the length of the course.** This number is the resolution of a
mystery the reader has been building toward since Volume I. In [§1.4](../01-elementary-mechanics/kepler-orbits.ipynb) we found, with the
conserved Laplace–Runge–Lenz vector, that an isolated two-body Kepler orbit is *exactly closed*
— a pure $1/r^2$ force makes no precession at all. In [§2.4](../02-classical-mechanics/central-force.ipynb) we found the mechanism: any
departure from $1/r^2$ turns the perihelion, at a rate we could compute, and an
inverse-fourth-power term in particular precesses the orbit by $6\pi GM/[a(1-e^2)c^2]$. We even
laid out Mercury's honest accounting there: of its $574''$/century, planetary tugs supply about
$531''$, leaving a stubborn $43''$ that no Newtonian effect explains. Here is where that
residual is explained. General relativity supplies exactly the inverse-fourth-power term the reader
already knew would precess the orbit — from the curvature of spacetime rather than from a fudge
— and the number it yields is the one astronomers measured. In the spirit of this notebook, the
reader predicted with their own integrator that such a term *must* precess the orbit; relativity
provides the term, and closes the thread.

## Exercise 8 — The effective potential, the ISCO, and the photon sphere

The special radii of a black hole come straight from a tool we have used since Volume I: the
effective potential. For Schwarzschild orbits it is $V_{\rm eff}(r)=(1-2M/r)(1+L^2/r^2)$
{eq}`eq-special-radii`, and its shape, controlled by the angular momentum $L$, organizes all
orbital behaviour ({numref}`fig-gr-veff`). As $L$ falls, the well that holds stable circular
orbits grows shallower, until at a critical $L^2=12M^2$ it has an inflection rather than a
minimum: that marginal orbit is the **innermost stable circular orbit** at $r=6M$, inside
which no stable orbit exists. The **photon sphere** at $r=3M$, where light itself can circle,
is the corresponding critical radius for null orbits.

**This exercise (worked).** Build $V_{\rm eff}(r)$ (reusing the effective-potential idea of
Volumes I and II), find the photon sphere as the root of the photon potential's derivative
with `scipy.optimize.brentq`, and find the ISCO as the radius minimizing the circular-orbit
angular momentum $L^2(r)=r^2/(r-3)$ via `numpy.gradient`. Confirm they are $6M$ and $3M$.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    np.array([r_ISCO, r_photon]),
    np.array([6.0, 3.0]),
    "the ISCO is at 6M and the photon sphere at 3M",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — Plunge versus stable orbit

The ISCO is a cliff edge, and we can drive a particle over it. Two test particles begin at
the same radius with the same radial speed, differing only in angular momentum. The one with
enough angular momentum is held out by the centrifugal barrier and settles into a precessing
orbit; the one with too little finds no barrier high enough, spirals inward, and **crosses
the horizon** at $r=2M$, captured forever ({numref}`fig-gr-plunge`). We catch the crossing
with a terminal `solve_ivp` event that stops the integration exactly at the horizon.

**This exercise (worked).** Integrate two orbits from $r=10M$ with a small radial velocity,
one with $L=4.6M$ and one with $L=3.4M$, using `scipy.integrate.solve_ivp` (DOP853) with a
terminal event at $r=2M$ (named `horizon_crossing`). Confirm the low-angular-momentum
particle triggers the event (plunges through the horizon) while the high-angular-momentum one
never does.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.check(
    len(orbits["plunge"].t_events[0]) > 0 and len(orbits["stable"].t_events[0]) == 0,
    "below a critical angular momentum the particle plunges through the horizon",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 10 — Light bending

Light is not exempt from curved spacetime. A ray follows a null geodesic $d^2u/d\phi^2+u=
3Mu^2$ {eq}`eq-light-bending`, and passing a mass with impact parameter $b$ it is bent by
$\Delta\phi\approx4M/b$ — exactly twice what a naive Newtonian "light has weight" argument
gives, the factor-of-two that made the 1919 eclipse measurement a decisive test. *As before,
take the null-geodesic equation as given and compute with it:* integrate the ray, find the
angle between its incoming and outgoing asymptotes, and compare to the prediction
({numref}`fig-gr-bending`).

**This exercise (student).** Integrate a ray with impact parameter $b=100M$ (weak field, so
the formula applies) from far away, starting at $u=0$ with $u'=1/b$, using
`scipy.integrate.solve_ivp` (DOP853) with a terminal event when the ray returns to $u=0$
(escapes to infinity). The total angle swept exceeds $\pi$ by the deflection; confirm it
matches $4M/b$.

In [ ]:
# (solution hidden on the public site)


### Validation 10

In [ ]:
validate.close(
    deflection,
    4.0 * M / b_impact,
    "light bends by 4M/b passing a mass — the 1919 eclipse test",
    rtol=5e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 11 — Gravitational lensing: the image of a black hole

The picture the public associates with a black hole — a dark shadow ringed by bright light,
the sky behind it smeared into arcs — is gravitational lensing, and we can compute it. *As
with the orbit and the light ray, take the bending physics as given and compute the image:*
every ray reaching the camera is a null geodesic, bent by the amount we just learned, and the
picture is what those bent rays bring from the background sky ({numref}`fig-gr-lensing`). Rays
aimed too close to the hole, with impact parameter $b<b_{\rm crit}=3\sqrt3\,M\approx5.196M$,
spiral in and never return — that is the central **shadow**. Rays just outside skim the photon
sphere many times, piling up into the bright **photon ring** at the shadow's edge, and rays
farther out carry a warped image of the sky behind.

**A note on method, and on engineering judgment.** How we compute this image is itself a
lesson in computational physics, because the obvious approach is the wrong one. We could trace
a geodesic *per pixel* — integrate the null equation for every one of the $400\times400$ rays
— but that is roughly $660\,$s for one frame, far too slow to rebuild in CI, and pure waste,
because the bending depends only on the impact parameter $b$. So we exploit that symmetry:
integrate a one-dimensional **deflection table** $\alpha(b)$ *once* (with `np.trapezoid` over
the turning-point integral, then `scipy.interpolate.interp1d` for the lookup), and **remap** a
background image through it with a single vectorized `numpy` operation. The whole image then
costs a couple of seconds on a CPU. Choosing the cheapest method that conveys the physics
faithfully is exactly the skill a practitioner exercises constantly; here is the ladder we
weighed:

- **Tier 1 — a lensed background (what we do).** Warp a background image through the
  deflection table. It produces the shadow, the photon ring, and the warped sky — the
  recognizable image — in $\sim2\,$s on a CPU, rebuilding cleanly in CI. The trick that makes
  it cheap is the radial symmetry: one 1-D table, then a remap, instead of a geodesic per
  pixel.
- **Tier 2 — add an accretion disk (a stretch; pointer).** The glowing disk warped above and
  below the hole, light from its far side bent over the top, is what makes the cinematic
  "Interstellar" frame. It is doable but a real escalation: a disk-emission model plus Doppler
  beaming and gravitational redshift for realism. We sketch it as a stretch and keep it out of
  the core, so the notebook stays focused.
- **Tier 3 — full per-pixel ray tracing and animation (out of scope; pointer).** A
  high-resolution rendered frame by brute-force per-pixel integration is minutes per frame, and
  an animation is hundreds of frames. *This* is where one reaches for a GPU. We do not do it
  in-notebook, precisely because it would dominate the build time.

The honest tradeoff: more realism costs more compute, and the practitioner's job is to spend
it where it buys physics. For going beyond Tier 1, per-pixel ray tracing is *embarrassingly
parallel* (each pixel is independent), which points to clear tools: **vectorize first** on the
CPU (batch the ODE over all pixels rather than Python-looping); reach for **GPU array
libraries** like `cupy` (a near-drop-in `numpy` on NVIDIA GPUs) or `jax` (with `jit`+`vmap` to
map the geodesic integrator over every pixel); compile per-ray integrators with `numba`
(`@njit`, or `@cuda.jit` for CUDA kernels); or use **semi-analytic shortcuts** (affine
transformations of the Schwarzschild geodesic, or elliptic-integral solutions) that make
real-time single-hole rendering feasible, with mature tools such as EinsteinPy available for
those who want to go further. Full Kerr (rotating) or multi-hole scenes remain expensive and
motivate GPU shaders or learned approximations.

**This exercise (worked).** Build the deflection table $\alpha(b)$ for $b>b_{\rm crit}$ with
`np.trapezoid` over the turning-point integral and `scipy.interpolate.interp1d`; remap a
checkerboard-and-stars background through it with a vectorized `numpy` operation, blacking out
the shadow where $b<b_{\rm crit}=3\sqrt3\,M$ and marking the photon ring; and display the
unlensed background beside the lensed image. Confirm the shadow radius $3\sqrt3\,M$ by
deriving it from the Exercise 8 photon sphere ($b=r/\sqrt{1-2M/r}$ at $r_{\rm ph}$), and
check the table's far end against the weak-field $4M/b$ of Exercise 10.

In [ ]:
# (solution hidden on the public site)


### Validation 11

In [ ]:
validate.close(
    b_crit_derived,
    b_crit,
    "the shadow radius 3√3 M follows from the measured photon sphere: r_ph/√(1−2M/r_ph)",
    rtol=1e-3,
)
validate.close(
    alpha_far,
    4.0 * M / 50.0,
    "the deflection table reproduces the weak-field 4M/b at its far end",
    rtol=8e-2,
)

In [ ]:
# (solution hidden on the public site)


### Validation 11b

In [ ]:
validate.check(
    bool(np.all(lensed[b_pix < b_crit] == 0.0) and lensed[b_pix > b_crit].max() > 0.0),
    "the image shows the shadow (captured rays, b<b_crit, are black) and the warped background outside",
)

## Exercise 12 — Gravity is geometry, and we computed it

We began Volume IV asking what happens at the speed of light, and we end it watching an orbit
precess around a black hole. The arc is complete. The equivalence principle made gravity a
matter of geometry, not force; gravitational time dilation followed, and your phone corrects
for it hourly. Then, taking the Schwarzschild metric and its geodesics as given — because the
full theory is a course we have not yet taken — we *computed* its signature predictions with
the very same ODE machinery used since Volume I: the precessing rosette, Mercury's $43''$, the
ISCO and photon sphere, a plunge through the horizon, and the bending of starlight. The free
particle's "extremize proper time" of [§4.7](relativistic-lagrangian-fields.ipynb) has become motion along geodesics of a curved
metric — and that is gravity itself. We could not derive general relativity, but we engaged
with it, computed real numbers from it, and found them right. That is the power and the
promise of computational physics: it lets us reach physics beyond our current theoretical
grasp, and learn from it honestly.

**This exercise.** Close Volume IV with a synthesis check: confirm with `np.allclose` that the
three headline numbers computed here — the Sun's Schwarzschild radius ($2.954\,$km), the ISCO
($6M$), and Mercury's precession ($43''$/century) — all match their known values at once, the
real predictions of general relativity that this notebook computed without deriving the
theory.

In [ ]:
# (solution hidden on the public site)


### Validation 12

In [ ]:
validate.check(
    all_correct,
    "general relativity's signature predictions — computed, not derived — all match observation",
)

## Notebook summary

This notebook closes **Volume IV**, carrying special relativity into its gravitational
sequel and computing genuine predictions of general relativity without deriving the full
theory — the computational machinery as the great equalizer.

- **The equivalence principle** {eq}`eq-equivalence`, {eq}`eq-redshift`: acceleration and
  gravity are locally indistinguishable, so gravity is geometry; the Pound–Rebka redshift is
  $gh/c^2=2.46\times10^{-15}$, and only the non-removable **tidal** field marks true curvature.
- **Gravitational time dilation** {eq}`eq-grav-time-dilation`: clocks tick at $\sqrt{1-2M/r}$
  ($0.82$ at $6M$, $0.58$ at $3M$, zero at the horizon); GPS satellites run $+46\,\mu$s/day
  fast (net $+38$ with the SR term), a daily engineering correction.
- **Nondimensionalization** {eq}`eq-schwarzschild`: working in units of $M$ exposes the true
  ratios, keeps every number $O(1)$ where floating point is accurate (the [§0.1](../00-foundations/floating-point.ipynb) callback), and
  round-trips through the Sun's $r_s=2.954\,$km.
- **The computed orbit** {eq}`eq-orbit`: integrating $d^2u/d\phi^2+u=M/L^2+3Mu^2$ gives a
  **precessing rosette** (the $3Mu^2$ term), and the round-trip to Mercury yields exactly
  $43''$/century — GR's first triumph.
- **Black-hole structure** {eq}`eq-special-radii`, {eq}`eq-light-bending`: the effective
  potential sets the **ISCO at $6M$** and **photon sphere at $3M$**; below a critical $L$ a
  particle **plunges through the horizon** (a terminal `solve_ivp` event); a ray bends by
  $4M/b$, the 1919 eclipse test; and remapping a background through a 1-D deflection table
  produces the lensed black-hole **image** — the shadow at $3\sqrt3\,M$ and the photon ring —
  cheaply, by exploiting the radial symmetry rather than tracing a geodesic per pixel.

**Volume IV as a whole** began with the crisis of the speed of light ([§4.1](crisis-and-postulates.ipynb)), derived the
Lorentz transformation and its consequences ([§4.2](lorentz-transformation.ipynb)), found the geometry of spacetime and the
four-vector toolkit ([§4.3](spacetime-minkowski.ipynb)), dissolved the paradoxes ([§4.4](paradoxes.ipynb)), built four-momentum and $E=mc^2$
([§4.5](four-momentum-energy.ipynb)), put conservation to work in collisions ([§4.6](collisions-decays.ipynb)), reunited mechanics and electrodynamics
in the covariant force ([§4.7](relativistic-lagrangian-fields.ipynb)), and ends here watching gravity bend an orbit — the free
particle's proper-time principle become the geometry of curved spacetime.

## Outlook

- **Gravitational waves.** Ripples in spacetime from inspiralling black holes, now heard by
  LIGO — a prose pointer beyond this taste.
- **Cosmology.** The expanding universe and the FLRW metric, the geometry of the cosmos as a
  whole.
- **The black-hole image, extended.** The Tier-1 lensed image computed here can be pushed to
  the cinematic accretion-disk render (Tier 2 — disk emission with Doppler beaming and
  gravitational redshift) and, with GPU acceleration (`cupy`/`jax`/`numba`) or learned
  geodesics, to full per-pixel and animated relativistic ray tracing (Tier 3) — the "photon
  ring" of the Event Horizon Telescope — a natural standalone computational-GR project.
- **Numerical relativity.** Solving Einstein's equations on a computer for the full two-body
  problem — a far pointer.
- **Cross-reference** [§4.7](relativistic-lagrangian-fields.ipynb) (extremize proper time becomes geodesic motion), [§0.1](../00-foundations/floating-point.ipynb) (floating point
  and nondimensionalization), 1.x/2.x (the effective potential), and [§2.9](../02-classical-mechanics/lagrange-points.ipynb) (the nondimensional
  CR3BP).

In [ ]:
from ecp.style import footer

footer()